# Naive 4c-SCF
In this notebook we will define a very rudimentary 4c-SCF definition for the $\textrm{He}$ atom. The idea is to translate almost directly (with some particularities) the regular Pople-Nesbet UHF equations and implementation, including non-diagonal terms in the Dirac-Fock Hamiltonian. For brevity, some iterations of this notebook have been trashed, and only the barebones functioning scf is conserved. This imples that instead of building iteration per iteration, this notebook is the result of generalizing the SCF for $n$ iterations. 

In [1]:
import numpy as np
from py_mods.src.SCF.linalg import transformation_matrix
from py_mods.src.external.DIRAC_ME import (
    build_4c_one_Fock_from_h5,
    build_S_V_W_T_from_h5,
    get_nuc_charge,
    full_eri_from_checkpoint,
)
from py_mods.src.SCF.scf_kernels import calc_p_matrix_comp

In [2]:
# load results
He_F0_eigvals = np.loadtxt("files/He_F_eigvals_1st_iter.dat")
He_F1_eigvals = np.loadtxt("files/He_F_eigvals_2nd_iter.dat")
He_F2_eigvals = np.loadtxt("files/He_F_eigvals_3rd_iter.dat")

# 1. SCF preparation
In the same fashion as the previous SCF kernels, it is necessary to actually prepare a "context" for the calculation. This includes: 
1. Integrals: 
    1. ERIs: Computed (at this point from Interest)
    2. Hamiltonian matrix elements: $V$, $S$, $W$, $T_{rel}$ extracted from checkpoint
2. Basis set definition: 
    1. GTO definition: $l$, $\mathbf{r}$, $\alpha$, $d$ extracted from checkpoint
    2. Large-Small component definition: $nL$, $nS$ ($n_{tot} - nL$)
3. Charge:
    1. Nuclear charge: $Z$ (extracted from checkpoint)
    2. Electron charge: $N$ defined. 
4. Occupation: occupation determinant computed from the number of electrons and the number of large and small components.

## 1.1 Integrals
We start loading and computing the integrals with the predefined helper functions. In this implementation, we are considering the hardcoded size of $nL$, but the routines to retrieve matrix elements will be updated to automatically determine the size of the large and small components. 

In [3]:
h5_filename = "files/He_checkpoint.h5"

F_0 = build_4c_one_Fock_from_h5(h5_filename)
S, V, W, T = build_S_V_W_T_from_h5(h5_filename)
nuc_charge = get_nuc_charge(h5_filename)
eri = full_eri_from_checkpoint(h5_filename)

nL = 9

However, here comes the first difference due to the DF hamiltonian: ERIs are not the full tensor anymore, but they are separated in **classess** depending on the involved components:

$$
(LL|LL) = \sum_{pqrs} (\phi_{q}^{L} \phi_{p}^{L} | \phi_{r}^{L} \phi_{s}^{L}) 
$$

$$
(LL|SS) = \sum_{pqrs} (\phi_{q}^{L} \phi_{p}^{L} | \phi_{r}^{S} \phi_{s}^{S})
$$

$$
(SS|LL) = \sum_{pqrs} (\phi_{q}^{S} \phi_{p}^{S} | \phi_{r}^{L} \phi_{s}^{L})
$$

$$
(SS|SS) = \sum_{pqrs} (\phi_{q}^{S} \phi_{p}^{S} | \phi_{r}^{S} \phi_{s}^{S})
$$

Where we recall that we use chemists notation in which: 

$$
(pq|rs) = \int \int \phi_{p}(\mathbf{r}_{1}) \phi_{q}(\mathbf{r}_{1}) \frac{1}{|\mathbf{r}_{1} - \mathbf{r}_{2}|} \phi_{r}(\mathbf{r}_{2}) \phi_{s}(\mathbf{r}_{2}) d\mathbf{r}_{1} d\mathbf{r}_{2}
$$

The difference between these classess and the use of the total ERI tensor is that integrals with mixed $LS$ components in either the bra or the ket are not  not needed for the construction of the Fock matrix due to the structure of the DF Hamiltonian:
$$
h_d = \begin{pmatrix}
h_d^{LL} & h_d^{LS} \\
h_d^{SL} & h_d^{SS}
\end{pmatrix}
$$

Where ${LL}$ and ${SS}$ only involve the $(LL|LL)$, and $(SS|SS)$ integrals, and ${LS}$ and ${SL}$ only involve the $(LL|SS)$ and $(SS|LL)$ integrals (this will be shown later when the formulas for $J$ and $K$ are presented). Therefore, since we will be using highly efficient tensor contraction routines, the "excess" $(LS|SS)$, ..., integrals will alter the matrix elements over the contraction, and thus must be removed. 

In this naive approach, the ERIs are computed interest file, and then they are grouped into the different classess over an empty ERI tensor, so that only $(LL|LL)$, $(LL|SS)$, $(SS|LL)$ and $(SS|SS)$ integrals are conserved:

In [4]:
# Filter ERI for nL block
eri_classess = np.zeros_like(eri)

eri_classess[:nL, :nL, :nL, :nL] = eri[:nL, :nL, :nL, :nL] # LL-LL block
eri_classess[:nL, :nL, nL:, nL:] = eri[:nL, :nL, nL:, nL:] # LL-SS block
eri_classess[nL:, nL:, :nL, :nL] = eri[nL:, nL:, :nL, :nL] # SS-LL block
eri_classess[nL:, nL:, nL:, nL:] = eri[nL:, nL:, nL:, nL:] # SS-SS block

eri = eri_classess

## 1.2 Occupation vector and X transformation matrix
### 1.2.1 Occupation vector
The same way as in regular occupation selected SCF (as has been the method for the non-relativistic counterpart), we need to define the occupation vector to contract over in order to build the density matrix. Since there is large and small components, it is important that we align the occupation so that negative energy solution are not occupied, and positive energy solutions are (for now) occupied in order. 

Since it is known that there are as many positive energy solutions as large component basis functions, we can define the occupation vector as:

In [5]:
occ = np.zeros(S.shape[0])

total_charge = 0  # since it is netural He
n_electrons = nuc_charge + total_charge
n_pos_states = 2 * nL

occ[-n_pos_states : -n_pos_states + n_electrons] = 1  # since it is occupied in order for now 

total_occ_det = occ

print(total_occ_det)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


However it should be straightforward to generalize this for any occupation vector as in the current non-relativistic implementation.

### 1.2.2 Transformation matrix
And the transformation matrix $X$ in order to transform the basis into the orthogonal representation: 

$$
h_dC = SCe
$$

And in the orthogonal representation:

$$
h_d^{orth} = X^{\dagger} h_d X
$$

And thus:
$$
h_d^{orth} C^{orth} = \mathbb{I} C^{orth} e
$$

Where the transformation matrix will be built using symmetric orthogonalization:

$$ 
X = S^{-\frac{1}{2}} = U s^{-\frac{1}{2}} U^{\dagger}
$$


In [6]:
X = transformation_matrix(S)

## 1.3 Matrix elements
### 1.3.1 One-electron part
We need to determine, as in regular SCF procedures the core Hamiltonian, which in this case is the one-electron part of the Dirac-Fock Hamiltonian, which is defined as:


$$
h_d^{[1]} = \begin{pmatrix}
\mathbf{V}_{\mu \nu}^{L\alpha L\alpha}  & -ic \mathbf{d}_{z}^{LS}  & \mathbf{0} &  -ic \mathbf{d}_{-}^{LS} \\
-ic \mathbf{d}_{z}^{SL} & \mathbf{W}_{\mu \nu}^{S\alpha S\alpha} & -ic \mathbf{d}_{-}^{SL}   & \mathbf{0}  \\
\mathbf{0} & -ic \mathbf{d}_{+}^{LS}  & \mathbf{V}_{\mu \nu}^{L\beta L\beta} & ic \mathbf{d}_{z}^{LS}  \\
-ic \mathbf{d}_{+}^{SL}   & \mathbf{0} & ic \mathbf{d}_{z}^{SL}  & \mathbf{W}_{\mu \nu}^{S\beta S\beta} 
\end{pmatrix}
$$

Which is built directly as

In [7]:
H_core = T + V + W

### 1.3.2 two-electron part

In the Two electron part, we distinguish as in regular Hartree-Fock theory two types of integrals:

$$
J_{\mu\nu} = \sum_{\lambda\sigma} P_{\lambda\sigma} (\mu\nu|\lambda\sigma)
$$

And 
$$
K_{\mu\nu} = \sum_{\lambda\sigma} P_{\lambda\sigma} (\mu\lambda|\nu\sigma)
$$

Where the density matrix is computed via the contraction with the occupation vector as:

$$
P_{\lambda\sigma} = \sum_{i} n_i C_{\lambda i} C_{\sigma i}^{*}
$$

The two electron part of the Fock matrix is then built as:

$$
h_d^{[2]} = \begin{pmatrix}
J^{L\alpha L\alpha} - K^{L\alpha L\alpha}  & - K^{L\alpha S\alpha}    & - K^{L\alpha L\beta}   & - K^{L\alpha S\beta}  \\
- K^{S\alpha L\alpha} & J^{S\alpha S\alpha} - K^{S\alpha S\alpha}  & K^{S\alpha L\beta}   & - K^{S\alpha S\beta} \\
- K^{L\beta L\alpha} & -K^{L\beta S\alpha} & J^{L\beta L\beta} - K^{L\beta L\beta}  & - K^{L\beta S\beta} \\
- K^{S\beta L\alpha} & -K^{S\beta S\alpha} & -K^{S\beta S\beta}  & J^{S\beta L\beta} - K^{S\beta S\beta}
\end{pmatrix}
$$

#### 1.3.2.1 $J$ integrals

The difference from the non-relativistic counterpart is that computation depends on the classess involved. In particular, in the case of $J$ integrals, only the diagonal $J$ blocks are nonzero. Thus, we define $LL$ and $SS$ classess as:

$$
J^{(LL)}_{\lambda \sigma} = \sum_{\mu\nu} P_{\mu\nu}^{(LL)} (\mu\nu|\lambda\sigma)^{(LL|LL)} + \sum_{\mu\nu} P_{\mu\nu}^{(SS)} (\mu\nu|\lambda\sigma)^{(LL|SS)}
$$

And:

$$
J^{(SS)}_{\lambda \sigma} = \sum_{\mu\nu} P_{\mu\nu}^{(LL)} (\mu\nu|\lambda\sigma)^{(SS|LL)} + \sum_{\mu\nu} P_{\mu\nu}^{(SS)} (\mu\nu|\lambda\sigma)^{(SS|SS)}
$$

Where the denisty matrix, analogous to UHF theory is defined as:

$$
P_{\mu\nu}^{(LL)} = P_{\mu\nu}^{(LL,\alpha)} + P_{\mu\nu}^{(LL,\beta)} 
$$ 

And:
$$
P_{\mu\nu}^{(SS)} = P_{\mu\nu}^{(SS,\alpha)} + P_{\mu\nu}^{(SS,\beta)}
$$

#### 1.3.2.2 $K$ integrals
In the case of the K integrals, all the different sub-blocks are non-zero, and thus we need to define the different classess as:

$$
K^{(pq)}_{\lambda \sigma} = \sum_{\mu\nu} P_{\mu\nu}^{(pq)} (\mu\lambda|\nu\sigma)^{(LL|LL)}, \quad \quad \quad \quad p, q = L\alpha, L\beta
$$


$$
K^{(pq)}_{\lambda \sigma} = \sum_{\mu\nu} P_{\mu\nu}^{(pq)} (\mu\lambda|\nu\sigma)^{(SS|SS)}, , \quad \quad \quad \quad p, q = S\alpha, S\beta
$$

And the off-diagonal terms as: 

$$
K^{(pq)}_{\lambda \sigma} = \sum_{\mu\nu} P_{\mu\nu}^{(pq)} (\mu\lambda|\nu\sigma)^{(LL|SS)} \quad \quad \quad \quad p = L\alpha, L\beta \quad; \quad q = S\alpha, S\beta
$$

For reference, see equations $94-102$ in chapter $2$ of "Methods in Computational Chemistry. Volume 2: Relativistic Effects in Atoms and Molecules". 

The previous formulas can be reduced to the following function, were we use einsum contractions instead of loops:

In [8]:
def g_matrix_4c(P, eri):
    n_bas = P.shape[0]
    n_bas_half = n_bas // 2

    P_aa = P[0:n_bas_half, 0:n_bas_half]
    P_bb = P[n_bas_half:n_bas, n_bas_half:n_bas]
    P_ab = P[0:n_bas_half, n_bas_half:n_bas]
    P_ba = P[n_bas_half:n_bas, 0:n_bas_half]

    P_total = P_aa + P_bb
    J = np.einsum("mnsl, ls -> mn", eri, P_total)

    K_aa = np.einsum("psrq, sr -> pq", eri, P_aa)
    K_bb = np.einsum("psrq, sr -> pq", eri, P_bb)
    K_ab = np.einsum("psrq, sr -> pq", eri, P_ab)
    K_ba = np.einsum("psrq, sr -> pq", eri, P_ba)

    G_aa = J - K_aa
    G_bb = J - K_bb
    G_ab = -K_ab
    G_ba = -K_ba

    G_full = np.zeros((n_bas, n_bas), dtype=np.complex128)
    # And we fill the matrix by blocks
    G_full[0:n_bas_half, 0:n_bas_half] = G_aa
    G_full[n_bas_half:n_bas, n_bas_half:n_bas] = G_bb
    G_full[0:n_bas_half, n_bas_half:n_bas] = G_ab
    G_full[n_bas_half:n_bas, 0:n_bas_half] = G_ba

    return G_full

# 2. SCF definition

We can borrow from regular SCF theory the workflow, as at this point it is exacly the same SCF iterations. We will start with the naivest implementation, as the prebuilt SCF routines should be straightforward to adapt. We recall that the SCF "internal part" is defined as:

1. Transformation to orthogonal basis
2. Diagonalization of the Fock matrix
3. (Optional) reorder the eigenvalues and eigenvectors if needed
4. Back transformation to original basis


In [9]:
def scf_iteration(F_1, X, total_occ_det):
    F_p1 = X.T @ F_1 @ X

    e1, w1 = np.linalg.eigh(F_p1)

    idx = np.argsort(e1)
    e1 = e1[idx]
    w1 = w1[:, idx]

    c_alpha_beta1 = X @ w1
    P_1 = calc_p_matrix_comp(c_alpha_beta1.conj().T, c_alpha_beta1, total_occ_det)

    return e1, w1, F_p1, P_1

While the "external" part of the scf is defined as: 
1. Build the Fock matrix from the core Hamiltonian and the two-electron part
2. "Internal" step
3. Build the density matrix
4. Compute the two electron part of the Fock matrix

In [10]:
def scf_steps(n_steps, H_core, eri, X, total_occ_det):
    energy_step = []
    P_old = np.zeros_like(H_core)

    for i in range(n_steps):
        if i == 0:
            G_new = np.zeros_like(H_core)
        else:
            G_new = g_matrix_4c(P_old, eri)

        if i > 0:
            e_scf = np.linalg.trace(P_old @ H_core + 0.5 * P_old @ G_new)
            energy_step.append(e_scf.real)

        F_new = H_core + G_new

        e_new, w_new, F_p_new, P_2 = scf_iteration(F_new, X, total_occ_det)

        if i == 0:
            e_scf = np.linalg.trace(P_2 @ H_core)
            energy_step.append(e_scf.real)

        P_old = P_2

    # print(energy_step)
    return energy_step

And convergence control is located elsewhere wrapping this logic. 

# 3. Results
We can finally put to the test this naive implementation and compare with the results obtained from DIRAC. In particular we will take the total energy as the reference metric at each iteration:

In [11]:
scf_energies = scf_steps(15, H_core, eri, X, total_occ_det)
scf_reference_energies = np.loadtxt("files/He_scf_energy.dat")

print(f"Difference at each iteration of SCF energies: {scf_energies[:len(scf_reference_energies)] - scf_reference_energies}")

Difference at each iteration of SCF energies: [-9.99684779e-11 -9.20152843e-13 -3.91686683e-13 -2.92210700e-13
  2.46025422e-13  2.12718732e-13 -4.24105195e-13 -2.13606910e-13
  1.55431223e-14  1.11022302e-14 -2.35367281e-14 -2.08721929e-14
 -2.39808173e-14]


Which looks fantastic. 